In [1]:
import os
%cd ../

/Users/arjunadhithyarr/Documents/z-arjun/numpy-neural-net-play


In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
from nn import *

In [3]:
# one neuron, target 1, loss = (out - target)^2
neuron = singleNeuron(w=[0.5, -1.0, 2.0], b=0.25, activation=sigmoid)
out = neuron.forward([1.0, 2.0, -1.0])
target = 1.0

dL_dout = 2 * (out - target)
dL_dx = single_neuron_backprop(neuron, dL_dout)

print(f"out      {float(out):.6f}")
print(f"dL_dout  {float(dL_dout):.6f}")
print(f"dL_dz    {float(neuron.dL_dz):.6f}")
print(f"dW       {np.round(neuron.dW, 6)}")
print(f"db       {float(neuron.db):.6f}")
print(f"dL_dx    {np.round(dL_dx, 6)}")

out      0.037327
dL_dout  -1.925346
dL_dz    -0.069185
dW       [-0.069185 -0.138369  0.069185]
db       -0.069185
dL_dx    [-0.034592  0.069185 -0.138369]


In [4]:
# nudge each weight by a tiny eps and watch the loss move
def neuron_loss(w):
    n = singleNeuron(w=w, b=0.25, activation=sigmoid)
    return (n.forward([1.0, 2.0, -1.0]) - target) ** 2

eps = 1e-6
w0 = np.array([0.5, -1.0, 2.0])
numeric_dW = np.array([
    (neuron_loss(w0 + eps * np.eye(3)[i]) - neuron_loss(w0 - eps * np.eye(3)[i])) / (2 * eps)
    for i in range(3)
])
print("analytic dW", np.round(neuron.dW, 6))
print("numeric  dW", np.round(numeric_dW, 6))

analytic dW [-0.069185 -0.138369  0.069185]
numeric  dW [-0.069185 -0.138369  0.069185]


## A layer of neurons

In [5]:
# two neurons with different weights, loss = mean squared error over both outputs
W = [
    [0.5, -1.0, 2.0],
    [1.0, 0.5, -0.5],
]
b = [0.25, 0.0]
x = [1.0, 2.0, -1.0]
targets = np.array([1.0, 0.0])

layer = multipleNeuron(n=2, n_inputs=3, activation=sigmoid, W=W, b=b)
outs = layer.forward(x)

dL_douts = 2 * (outs - targets) / len(outs)
dL_dx = multi_neuron_backprop(layer, dL_douts)

print("outs   ", np.round(outs, 6))
print("loss   ", round(float(mean_squared_error(targets, outs)), 6))
for i, n in enumerate(layer.neurons):
    print(f"neuron {i}  dW {np.round(n.dW, 6)}  db {float(n.db):.6f}")
print("dL_dx  ", np.round(dL_dx, 6))

outs    [0.037327 0.924142]
loss    0.890389
neuron 0  dW [-0.034592 -0.069185  0.034592]  db -0.034592
neuron 1  dW [ 0.064786  0.129572 -0.064786]  db 0.064786
dL_dx   [ 0.04749   0.066985 -0.101577]


In [6]:
# dL_dx is what the layer before this one would receive; check it numerically too
def layer_loss(x_in):
    l = multipleNeuron(n=2, n_inputs=3, activation=sigmoid, W=W, b=b)
    return mean_squared_error(targets, l.forward(x_in))

x0 = np.array(x)
numeric_dx = np.array([
    (layer_loss(x0 + eps * np.eye(3)[i]) - layer_loss(x0 - eps * np.eye(3)[i])) / (2 * eps)
    for i in range(3)
])
print("analytic dL_dx", np.round(dL_dx, 6))
print("numeric  dL_dx", np.round(numeric_dx, 6))

analytic dL_dx [ 0.04749   0.066985 -0.101577]
numeric  dL_dx [ 0.04749   0.066985 -0.101577]
